# 2262 steps 6-10: models, scoring and verdict

Compares models of the employment status shares against the pipeline's rolling average (**as-is**), with each workplace feature added on its own and then together:

- **Multinomial logit** (candidate): log rolling shares and role, fitted to the imputed shares as fractional outcomes.
- **Lasso**, one per status (candidate): same inputs, clipped and renormalised to sum to 1.
- **Gradient boosting** (accuracy ceiling only, never a candidate): all features.

One pooled model across roles, trained on the imputed shares (`*_pct_imputed`) of a sample of workplaces and scored on **known** shares of
held-out workplaces (5 location folds, each scored by a model that never saw its workplaces). The rolling average is a feature and is
not fold-safe, so as-is is the like-for-like comparison. The guard re-runs each model on data with each workplace's last 12 months hidden.

**Win:** at least 1pp lower held-out headline error than as-is **and** better in all 5 folds, **and** not worse on trend or on the hidden-months guard.
Jumpiness is reported for every variant and highlighted only when it degrades notably relative to the gain, or as a tie-breaker.

**Before running:** `01_dataset_and_baselines.ipynb` has passed its benchmark gate. Writes only `results/model_scores.parquet`,
`results/all_scores.parquet` and `results/trend_series.parquet`.

**Stop the notebook instance when you finish** - SageMaker bills while it is running.

## Parameters

Same as notebook 1, plus:

- `TRAIN_PERCENT` - share of workplaces (outside the jumpiness sample) used to train; `MAX_TRAIN_ROWS` caps rows per fit.
- `LOG_FLOOR` - floor before taking logs of rolling shares. `TIME_REGISTERED_CAP_MONTHS` - cap on the time-registered input.
- `LOGIT_*`, `LASSO_*`, `HGB_MAX_ITER`, `RANDOM_SEED` - model settings.
- `FEATURE_SETS` - the ablation: no feature, each on its own, then all. `CATEGORICAL_LEVELS` - one-hot levels per feature.
- `WIN_PP`, `JUMPINESS_FLAG_FACTOR`, `TIE_PP` - the win rule. A variant is flagged when its workplace jumpiness rises by more than `JUMPINESS_FLAG_FACTOR` times its percentage error gain; variants within `TIE_PP` of the best headline are shown as ties.

In [ ]:
REPO_DIR = "/home/ec2-user/SageMaker/DataEngineering"  # clone made by sagemaker_setup.ipynb, on main
DATA_BUCKET = "sfc-2262-employment-datasets"
RESULTS_PREFIX = "s3://skillsforcare-nonprod/sagemaker-notebooks/empstat-model-compare/"
STORAGE_OPTIONS = None  # e.g. {"aws_region": "eu-west-2"} if Polars can't find the region
LOCAL = False  # True when RESULTS_PREFIX is a local folder

SHARES_PATH = f"s3://{DATA_BUCKET}/domain=03_ind_cqc/dataset=02_employment_status_03_impute/"
ESTIMATES_PATH = f"s3://{DATA_BUCKET}/domain=03_ind_cqc/dataset=01_filled_posts_05_estimated/"
RATINGS_PATH = f"s3://{DATA_BUCKET}/domain=02_sfc/dataset=sfc_cqc_ratings_for_data_requests/"
MODELLING_DATA_PATH = f"{RESULTS_PREFIX}modelling_dataset.parquet"
HIDDEN_TREND_PATH = f"{RESULTS_PREFIX}hidden_trendline.parquet"
HIDDEN_DATA_PATH = f"{RESULTS_PREFIX}hidden_dataset.parquet"
RESULTS_DIR = f"{RESULTS_PREFIX}results/"

N_FOLDS = 5
FOLD_SEED = 42
SAMPLE_PERCENT = 10  # workplaces used for jumpiness and the hidden-months test
SAMPLE_SEED = 42
HIDDEN_MONTHS = 12
ROLLING_AVERAGE_PERIOD = "6mo"
EXTRAPOLATION_PERIOD = "2y"
INTERPOLATION_CAP_PERIOD = "5y"
FY_START_MONTH = 4  # headline years are financial years starting this month

# Column names are defined once, here.
LOCATION = "locationId"
DATE = "cqc_location_import_date"
SERVICE = "primary_service_type"
REGION = "current_Region"
CSSR = "current_CSSR"
ROLE = "published_job_role_label"
WORKERS = "estimate_filled_posts_by_job_role"
POSTS = "estimate_filled_posts"
TIME_REGISTERED = "time_registered"  # months
DORMANCY = "dormancy"
TIME_SINCE_DORMANT = "time_since_dormant"
RATING_DATE = "rating_date"
OVERALL_RATING = "Overall_rating"
STATUSES = ["permanent", "temporary", "bank_or_pool", "agency", "other"]
KNOWN = [f"ascwds_emp_stat_{status}_pct" for status in STATUSES]
TREND = [f"{column}_imputed_trendline" for column in KNOWN]
ROLLING = [f"{column}_rolling_avg" for column in KNOWN]
IMPUTED = [f"{column}_imputed" for column in KNOWN]
TRUTH = [f"{column}_truth" for column in KNOWN]
FOLD_SAFE = [f"{column}_fold_safe" for column in ROLLING]
HIDDEN = "hidden"
FOLD = "fold"
CELL_KEYS = [SERVICE, REGION, ROLE]

# Features built here.
SIZE_BAND = "size_band"
DORMANCY_LEVEL = "dormancy_level"
RATING = "cqc_rating"
DORMANCY_RECORDED = "dormancy_recorded"
SIZE_BAND_EDGES = [(10, "<10"), (50, "10-49"), (150, "50-149")]  # upper edges, exclusive
SIZE_BAND_TOP = "150+"
SIZE_BAND_UNKNOWN = "unknown"
SIZE_BANDS = [label for _, label in SIZE_BAND_EDGES] + [SIZE_BAND_TOP, SIZE_BAND_UNKNOWN]
DORMANT_VALUE = "Y"
DORMANCY_NOT_RECORDED = "not recorded"
DORMANCY_NONE_YET = "none yet"
DORMANT_NOW = "dormant now"
DORMANT_RECENT = "dormant 1-12m ago"
DORMANT_LONG = "dormant 13m+ ago"
DORMANT_RECENT_MONTHS = 12
DORMANCY_LEVELS = [DORMANCY_NOT_RECORDED, DORMANCY_NONE_YET, DORMANT_NOW, DORMANT_RECENT, DORMANT_LONG]
RATINGS_BEST_FIRST = ["Outstanding", "Good", "Requires improvement", "Inadequate"]
UNRATED = "unrated"
RATING_LEVELS = RATINGS_BEST_FIRST + [UNRATED]

# Result column and metric names.
VARIANT, TEST, LEVEL, GROUP, METRIC, VALUE = "variant", "test", "level", "group", "metric", "value"
HELD_OUT, GUARD = "held_out", "hidden_months"
ROWS, MAE_PP, BRIER, TOP1 = "number_of_rows", "mae_pp", "brier_score", "top_1_accuracy"
MAE_SKILL, BRIER_SKILL = "mae_skill_score", "brier_skill_score"
BASELINE_MAE, BASELINE_BRIER = "baseline_mae", "baseline_brier_score"
TREND_MAE_PP, MEAN_CHANGE = "trend_mae_pp", "mean_period_to_period_change"
CONTRIBUTORS, WINDOW_CONTRIBUTORS, YEAR = "contributing_workplaces", "window_contributing_workplaces", "financial_year"

TRAIN_PERCENT = 20
MAX_TRAIN_ROWS = 400_000
LOG_FLOOR = 1e-3
TIME_REGISTERED_CAP_MONTHS = 120
LOGIT_C = 1.0
LOGIT_MAX_ITER = 300
LASSO_N_ALPHAS = 10
LASSO_CV = 3
LASSO_MAX_ITER = 1000
HGB_MAX_ITER = 200
RANDOM_SEED = 42

FEATURE_SETS = {
    "base": [],
    "time_registered": [TIME_REGISTERED],
    "size_band": [SIZE_BAND],
    "dormancy": [DORMANCY_LEVEL],
    "rating": [RATING],
    "all": [TIME_REGISTERED, SIZE_BAND, DORMANCY_LEVEL, RATING],
}
CATEGORICAL_LEVELS = {SIZE_BAND: SIZE_BANDS, DORMANCY_LEVEL: DORMANCY_LEVELS, RATING: RATING_LEVELS}
PREDICTED = [f"{status}_predicted" for status in STATUSES]
BASELINES = ["as_is", "fold_safe"]
CEILING_MODEL = "hgb"
WIN_PP = 1.0
JUMPINESS_FLAG_FACTOR = 2.0
TIE_PP = 0.1

## Setup

In [ ]:
import sys

import polars as pl

sys.path.insert(0, REPO_DIR)
import projects._03_independent_cqc._02_employment_status.fargate.utils.impute_utils as imputeUtils
from projects._03_independent_cqc.utils import model_evaluation_utils as evaluation

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)

assert imputeUtils.PCT_COLUMNS == KNOWN and imputeUtils.IMPUTED_TRENDLINE_PCT_COLUMNS == TREND
assert imputeUtils.ROLLING_AVG_PCT_COLUMNS == ROLLING and imputeUtils.IMPUTED_PCT_COLUMNS == IMPUTED

from datetime import date

import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LassoCV, LogisticRegression

## Input and output helpers

In [ ]:
import io
from pathlib import Path


def scan(path: str) -> pl.LazyFrame:
    return pl.scan_parquet(path, storage_options=STORAGE_OPTIONS)


def sink(lf: pl.LazyFrame, path: str) -> None:
    lf.sink_parquet(path, storage_options=STORAGE_OPTIONS)


def write_result(df: pl.DataFrame, name: str) -> None:
    """Write a small result as parquet under RESULTS_DIR."""
    if LOCAL:
        path = Path(RESULTS_DIR) / name
        path.parent.mkdir(parents=True, exist_ok=True)
        df.write_parquet(path)
        return
    import boto3

    bucket, key = RESULTS_DIR.removeprefix("s3://").split("/", 1)
    buffer = io.BytesIO()
    df.write_parquet(buffer)
    boto3.client("s3").put_object(Bucket=bucket, Key=f"{key.rstrip('/')}/{name}", Body=buffer.getvalue())


def read_result(name: str) -> pl.DataFrame:
    """Read a result written by `write_result`."""
    if LOCAL:
        return pl.read_parquet(Path(RESULTS_DIR) / name)
    import boto3

    bucket, key = RESULTS_DIR.removeprefix("s3://").split("/", 1)
    body = boto3.client("s3").get_object(Bucket=bucket, Key=f"{key.rstrip('/')}/{name}")["Body"].read()
    return pl.read_parquet(io.BytesIO(body))


in_sample = pl.col(LOCATION).hash(SAMPLE_SEED) % 100 < SAMPLE_PERCENT

## Scoring functions (as notebook 1)

Same functions and checks as notebook 1, so every variant is scored exactly like the baselines.

In [ ]:
def calculate_mean_absolute_error(estimates: list[pl.Expr], known_columns: list[str]) -> pl.Expr:
    return pl.mean_horizontal((estimate - pl.col(known)).abs() for estimate, known in zip(estimates, known_columns))


def calculate_summed_squared_error(estimates: list[pl.Expr], known_columns: list[str]) -> pl.Expr:
    return pl.sum_horizontal((estimate - pl.col(known)) ** 2 for estimate, known in zip(estimates, known_columns))


def score_shares(
    lf: pl.LazyFrame, predicted_columns: list[str], known_columns: list[str], by_columns: list[str] | None = None
) -> pl.LazyFrame:
    """
    Score predicted against known shares row by row, every row counting equally.

    Only rows with every known and predicted share are scored. The skill scores compare each error with
    a baseline predicting the pooled mean known share on every row. A tied prediction counts as its first
    column for top 1.
    """
    by_columns = by_columns or []
    predicted = [pl.col(column) for column in predicted_columns]
    baseline = [pl.col(column).mean() for column in known_columns]
    scored_lf = lf.filter(pl.all_horizontal(pl.col(known_columns + predicted_columns).is_not_null()))
    row_scores_lf = scored_lf.select(
        *by_columns,
        calculate_mean_absolute_error(predicted, known_columns).alias(MAE_PP),
        calculate_summed_squared_error(predicted, known_columns).alias(BRIER),
        (pl.concat_list(predicted_columns).list.arg_max() == pl.concat_list(known_columns).list.arg_max())
        .cast(pl.Float64)
        .alias(TOP1),
        calculate_mean_absolute_error(baseline, known_columns).alias(BASELINE_MAE),
        calculate_summed_squared_error(baseline, known_columns).alias(BASELINE_BRIER),
    )
    mae = pl.col(MAE_PP).mean()
    brier = pl.col(BRIER).mean()
    scores = [
        pl.len().alias(ROWS),
        (mae * 100).alias(MAE_PP),
        brier,
        pl.col(TOP1).mean(),
        (1 - mae / pl.col(BASELINE_MAE).mean()).alias(MAE_SKILL),
        (1 - brier / pl.col(BASELINE_BRIER).mean()).alias(BRIER_SKILL),
    ]
    return row_scores_lf.group_by(by_columns).agg(scores) if by_columns else row_scores_lf.select(scores)


def long_scores(df: pl.DataFrame, level: str, group_column: str | None = None) -> pl.DataFrame:
    index = [group_column] if group_column else []
    group = (pl.col(group_column).cast(pl.String) if group_column else pl.lit("all")).alias(GROUP)
    return (
        df.with_columns(pl.all().exclude(index).cast(pl.Float64))
        .unpivot(index=index, variable_name=METRIC, value_name=VALUE)
        .select(pl.lit(level).alias(LEVEL), group, METRIC, VALUE)
    )


def complete_rows(predicted: list[str], known: list[str]) -> pl.Expr:
    return pl.all_horizontal(pl.col(predicted + known).is_not_null())


def headline_errors(
    df: pl.DataFrame, predicted: list[str], known: list[str], geography: str, by_fold: bool = False
) -> pl.DataFrame:
    """
    Mean absolute error (pp) of predicted against known shares, worker-weighted within each service,
    geography, role and financial year, then averaged over those cells.
    """
    weight = pl.col(WORKERS).cast(pl.Float64).fill_null(0)
    year = (pl.col(DATE).dt.year() - (pl.col(DATE).dt.month() < FY_START_MONTH).cast(pl.Int32)).alias(YEAR)
    keys = [SERVICE, geography, ROLE, year] + ([FOLD] if by_fold else [])
    cells_lf = (
        df.lazy()
        .filter(complete_rows(predicted, known) & (weight > 0))
        .group_by(keys)
        .agg((pl.col(column).cast(pl.Float64) * weight).sum() / weight.sum() for column in predicted + known)
    )
    error = pl.mean_horizontal((pl.col(p) - pl.col(k)).abs() for p, k in zip(predicted, known))
    scores = [pl.len().alias(ROWS), (error.mean() * 100).alias(MAE_PP)]
    if by_fold:
        return cells_lf.group_by(FOLD).agg(scores).collect()
    return cells_lf.select(scores).collect()


def aggregate_error(df: pl.DataFrame, predicted: list[str], known: list[str]) -> pl.DataFrame:
    """Mean absolute error (pp) of the mean predicted against the mean known share per service, region, role and date."""
    cells_lf = (
        df.lazy()
        .filter(complete_rows(predicted, known))
        .group_by(SERVICE, REGION, ROLE, DATE)
        .agg(pl.col(predicted).mean(), pl.col(known).mean())
    )
    error = pl.mean_horizontal((pl.col(p) - pl.col(k)).abs() for p, k in zip(predicted, known))
    return cells_lf.select(pl.len().alias(ROWS), (error.mean() * 100).alias(MAE_PP)).collect()


def national_trend(df: pl.DataFrame, predicted: list[str], known: list[str]) -> pl.DataFrame:
    """Mean predicted and known share per date and status, in long format."""
    dates_lf = df.lazy().filter(complete_rows(predicted, known)).group_by(DATE).agg(pl.col(predicted).mean(), pl.col(known).mean())
    return pl.concat(
        [
            dates_lf.select(DATE, pl.lit(status).alias("status"), pl.col(p).alias("predicted_mean"), pl.col(k).alias("known_mean"))
            for status, p, k in zip(STATUSES, predicted, known)
        ]
    ).collect()


def jumpiness_scores(jitter_df: pl.DataFrame, predicted: list[str]) -> pl.DataFrame:
    """Mean absolute change between dates (own units) per workplace and role, and per service, region and role."""
    aggregate_lf = jitter_df.lazy().group_by(SERVICE, REGION, ROLE, DATE).agg(pl.col(predicted).mean())
    results = []
    for level, lf, partition in [
        ("jumpiness_workplace", jitter_df.lazy(), [LOCATION, ROLE]),
        ("jumpiness_aggregate", aggregate_lf, CELL_KEYS),
    ]:
        change_df = evaluation.mean_period_to_period_change(lf, predicted, partition, DATE).collect()
        results.append(
            pl.DataFrame({LEVEL: [level], GROUP: ["all"], METRIC: [MEAN_CHANGE], VALUE: [change_df[MEAN_CHANGE].mean()]})
        )
    return pl.concat(results)


def score_variant(
    variant: str, test: str, df: pl.DataFrame, predicted: list[str], known: list[str], jitter_df: pl.DataFrame
) -> pl.DataFrame:
    """
    Score one variant in long format.

    `df` holds the rows scored (known rows, or the hidden rows); `jitter_df` holds every row of the
    sampled workplaces, for jumpiness.
    """
    df = df.with_columns(
        pl.when(pl.col(DORMANCY_LEVEL) == DORMANCY_NOT_RECORDED)
        .then(pl.lit("not recorded"))
        .otherwise(pl.lit("recorded"))
        .alias(DORMANCY_RECORDED)
    )
    results = []
    for level, by in [("share_pooled", None), ("share_size_band", SIZE_BAND), ("share_dormancy", DORMANCY_RECORDED)]:
        scores_df = score_shares(df.lazy(), predicted, known, [by] if by else None).collect()
        results.append(long_scores(scores_df, level, by))
    results.append(long_scores(headline_errors(df, predicted, known, REGION), "headline"))
    results.append(long_scores(headline_errors(df, predicted, known, CSSR), "headline_cssr"))
    results.append(long_scores(headline_errors(df, predicted, known, REGION, by_fold=True), "headline_fold", FOLD))
    results.append(long_scores(aggregate_error(df, predicted, known), "aggregate"))
    trend_df = national_trend(df, predicted, known)
    trend_error = (trend_df["predicted_mean"] - trend_df["known_mean"]).abs().mean() * 100
    results.append(pl.DataFrame({LEVEL: ["trend"], GROUP: ["all"], METRIC: [TREND_MAE_PP], VALUE: [trend_error]}))
    results.append(jumpiness_scores(jitter_df, predicted))
    return pl.concat(results).with_columns(pl.lit(variant).alias(VARIANT), pl.lit(test).alias(TEST))


def metric_value(scores_df: pl.DataFrame, variant: str, test: str, level: str, metric: str, group: str = "all") -> float:
    row = scores_df.filter(
        (pl.col(VARIANT) == variant) & (pl.col(TEST) == test) & (pl.col(LEVEL) == level)
        & (pl.col(METRIC) == metric) & (pl.col(GROUP) == group)
    )
    return row[VALUE].item()

## Hidden-months functions (as notebook 1)

In [ ]:
def hide_last_months(lf: pl.LazyFrame, months: int = HIDDEN_MONTHS) -> pl.LazyFrame:
    """
    Null each workplace's known percentages in its last `months` months with known values,
    keeping the originals in "<column>_truth" and flagging the hidden rows in "hidden".
    """
    last_known_date = pl.col(DATE).filter(pl.col(KNOWN[0]).is_not_null()).max().over(LOCATION)
    hidden = pl.col(DATE) > last_known_date.dt.offset_by(f"-{months}mo")
    return lf.with_columns(
        *[pl.col(column).alias(truth) for column, truth in zip(KNOWN, TRUTH)],
        hidden.fill_null(False).alias(HIDDEN),
    ).with_columns(pl.when(pl.col(HIDDEN)).then(None).otherwise(pl.col(column)).alias(column) for column in KNOWN)


def impute_hidden(df: pl.DataFrame) -> pl.DataFrame:
    """Run the pipeline's imputation, which fills gaps along the ROLLING columns."""
    return imputeUtils.add_imputed_pcts(df.lazy()).collect(engine="streaming")

## Load the data

- `known_df` - every row with known shares: the scored rows.
- `sample_df` - every row of the jumpiness sample (scored for jumpiness only).
- `train_df` - rows with imputed shares for a different random sample of workplaces, used for fitting. Fits only use workplaces outside the fold being scored.

In [ ]:
modelling_lf = scan(MODELLING_DATA_PATH)
ROW_COLUMNS = [LOCATION, SERVICE, REGION, CSSR, ROLE, DATE, WORKERS, FOLD, SIZE_BAND, DORMANCY_LEVEL, TIME_REGISTERED, RATING]
hash_percent = pl.col(LOCATION).hash(SAMPLE_SEED) % 100
in_train_sample = (hash_percent >= SAMPLE_PERCENT) & (hash_percent < SAMPLE_PERCENT + TRAIN_PERCENT)

known_df = (
    modelling_lf.filter(pl.all_horizontal(pl.col(KNOWN).is_not_null()))
    .select(*ROW_COLUMNS, *KNOWN, *ROLLING)
    .collect(engine="streaming")
)
sample_df = modelling_lf.filter(in_sample).select(*ROW_COLUMNS, *ROLLING).collect(engine="streaming")
train_df = (
    modelling_lf.filter(in_train_sample & pl.col(IMPUTED[0]).is_not_null())
    .select(*ROW_COLUMNS, *ROLLING, *IMPUTED)
    .collect(engine="streaming")
)
roles = sorted(modelling_lf.select(pl.col(ROLE).unique()).collect()[ROLE].to_list())
print(f"known {known_df.height:,}, sample {sample_df.height:,}, train {train_df.height:,}, roles {len(roles)}")

## Model inputs and fitting

`design` builds the inputs: log rolling shares, role, and the chosen features (time registered as a capped scale plus a missing flag; the others one-hot).
A missing rolling share is treated as even (0.2).

- **Logit:** each row is stacked once per status with the share as its weight, which fits a multinomial logit to fractional outcomes.
- **Lasso:** `LassoCV` per status on the share, clipped to 0-1 and renormalised to sum to 1. A row clipped to all zeros keeps its rolling average.
- **Gradient boosting:** `HistGradientBoostingRegressor` per status, same handling. Accuracy ceiling only.

The check fits tiny known-answer data: the rolling average itself as target must be reproduced, and predictions must sum to 1.

In [ ]:
def rolling_matrix(df: pl.DataFrame) -> np.ndarray:
    return df.select(pl.col(ROLLING).fill_null(1 / len(STATUSES))).to_numpy().astype(np.float32)


def one_hot(series: pl.Series, levels: list[str]) -> np.ndarray:
    text = series.cast(pl.String)
    return np.column_stack([(text == level).to_numpy() for level in levels])


def design(df: pl.DataFrame, features: list[str]) -> np.ndarray:
    """Model inputs: log rolling shares, role, then each chosen feature."""
    parts = [np.log(np.clip(rolling_matrix(df), LOG_FLOOR, 1.0)), one_hot(df[ROLE], roles)]
    if TIME_REGISTERED in features:
        months = df[TIME_REGISTERED].cast(pl.Float64)
        scaled = months.clip(upper_bound=TIME_REGISTERED_CAP_MONTHS).fill_null(0) / TIME_REGISTERED_CAP_MONTHS
        parts.append(np.column_stack([scaled.to_numpy(), months.is_null().to_numpy()]))
    for column, levels in CATEGORICAL_LEVELS.items():
        if column in features:
            parts.append(one_hot(df[column], levels))
    return np.hstack(parts).astype(np.float32)


def normalise(predicted: np.ndarray, fallback: np.ndarray) -> np.ndarray:
    """Clip to 0-1 and rescale each row to sum to 1; a row clipped to all zeros keeps its fallback."""
    clipped = np.clip(predicted, 0, 1)
    total = clipped.sum(axis=1, keepdims=True)
    return np.where(total > 0, clipped / np.where(total > 0, total, 1), fallback).astype(np.float32)


def fit_model(model_name: str, X: np.ndarray, y: np.ndarray):
    """Fit one model to the shares `y`; returns a function giving raw predicted shares for new inputs."""
    if model_name == "logit":
        n_rows = len(X)
        weights = y.T.reshape(-1)
        keep = weights > 0
        model = LogisticRegression(C=LOGIT_C, max_iter=LOGIT_MAX_ITER)
        model.fit(np.tile(X, (len(STATUSES), 1))[keep], np.repeat(np.arange(len(STATUSES)), n_rows)[keep], sample_weight=weights[keep])
        return model.predict_proba
    if model_name == "lasso":
        models = [LassoCV(cv=LASSO_CV, n_alphas=LASSO_N_ALPHAS, max_iter=LASSO_MAX_ITER).fit(X, y[:, i]) for i in range(len(STATUSES))]
    else:
        models = [
            HistGradientBoostingRegressor(max_iter=HGB_MAX_ITER, random_state=RANDOM_SEED).fit(X, y[:, i])
            for i in range(len(STATUSES))
        ]
    return lambda new_X: np.column_stack([model.predict(new_X) for model in models])


rng = np.random.default_rng(RANDOM_SEED)
check_roll = rng.dirichlet(np.full(len(STATUSES), 5.0), 2000).astype(np.float32)
check_df = pl.DataFrame({ROLE: ["a"] * 2000, **{column: check_roll[:, i] for i, column in enumerate(ROLLING)}})
real_roles, roles = roles, ["a"]
check_X = design(check_df, [])
for name in ["logit", "lasso"]:
    check_pred = normalise(fit_model(name, check_X, check_roll)(check_X), check_roll)
    assert np.allclose(check_pred.sum(axis=1), 1, atol=1e-4), name
    assert np.abs(check_pred - check_roll).mean() < 0.03, (name, np.abs(check_pred - check_roll).mean())
roles = real_roles
zero_rows = normalise(np.zeros((1, len(STATUSES))), np.full((1, len(STATUSES)), 0.2))
assert np.allclose(zero_rows, 0.2)
print("model checks passed")

## Held-out workplaces

Per feature set, per model: for each fold, fit on the training workplaces outside the fold (at most `MAX_TRAIN_ROWS` rows), predict the fold's known rows and
jumpiness-sample rows, then score all folds' predictions together. Gradient boosting is run once, with all features. This is the cell that takes the time.

In [ ]:
MODELS_PER_SET = ["logit", "lasso"]
rng = np.random.default_rng(RANDOM_SEED)
known_folds, sample_folds, train_folds = (df[FOLD].to_numpy() for df in (known_df, sample_df, train_df))
known_rolling, sample_rolling = rolling_matrix(known_df), rolling_matrix(sample_df)
train_shares = train_df.select(IMPUTED).to_numpy().astype(np.float32)

model_scores, trend_series = [], []
for set_name, features in FEATURE_SETS.items():
    X_known, X_sample, X_train = (design(df, features) for df in (known_df, sample_df, train_df))
    models = MODELS_PER_SET + ([CEILING_MODEL] if set_name == "all" else [])
    for model_name in models:
        known_pred = np.full(known_rolling.shape, np.nan, dtype=np.float32)
        sample_pred = np.full(sample_rolling.shape, np.nan, dtype=np.float32)
        for fold in range(N_FOLDS):
            candidates = np.flatnonzero(train_folds != fold)
            chosen = rng.choice(candidates, size=min(MAX_TRAIN_ROWS, len(candidates)), replace=False)
            predict = fit_model(model_name, X_train[chosen], train_shares[chosen])
            for X, rolling, folds, out in [(X_known, known_rolling, known_folds, known_pred), (X_sample, sample_rolling, sample_folds, sample_pred)]:
                rows = np.flatnonzero(folds == fold)
                out[rows] = normalise(predict(X[rows]), rolling[rows])
        name = f"{model_name}_{set_name}"
        scored_df = known_df.hstack(pl.DataFrame(known_pred, schema=PREDICTED))
        jitter_df = sample_df.hstack(pl.DataFrame(sample_pred, schema=PREDICTED))
        model_scores.append(score_variant(name, HELD_OUT, scored_df, PREDICTED, KNOWN, jitter_df))
        trend_series.append(national_trend(scored_df, PREDICTED, KNOWN).with_columns(pl.lit(name).alias(VARIANT)))
        print("scored", name)
trend_series.append(national_trend(known_df, ROLLING, KNOWN).with_columns(pl.lit("as_is").alias(VARIANT)))

## Hidden months guard

Every workplace's last 12 months are hidden at once (notebook 1 wrote the data). Each model is fitted on the hidden-data imputed shares of the training workplaces,
predicts every row of the jumpiness sample, and those predictions replace the rolling average in the pipeline's imputation. Imputed shares are scored on the hidden
rows against the truth. One fit per variant (no folds), so workplaces in the scored sample are never in the training sample. Features are not re-dated to the cut-off.

In [ ]:
hidden_lf = scan(HIDDEN_DATA_PATH)
guard_columns = [*ROW_COLUMNS, HIDDEN, *KNOWN, *TRUTH, *ROLLING]
guard_sample_df = hidden_lf.filter(in_sample).select(guard_columns).collect(engine="streaming")
guard_train_df = impute_hidden(hidden_lf.filter(in_train_sample).select(guard_columns).collect(engine="streaming"))
guard_train_df = guard_train_df.filter(pl.col(IMPUTED[0]).is_not_null())
guard_rolling = rolling_matrix(guard_sample_df)
guard_shares = guard_train_df.select(IMPUTED).to_numpy().astype(np.float32)

for set_name, features in FEATURE_SETS.items():
    X_train, X_sample = design(guard_train_df, features), design(guard_sample_df, features)
    models = MODELS_PER_SET + ([CEILING_MODEL] if set_name == "all" else [])
    for model_name in models:
        chosen = rng.choice(len(X_train), size=min(MAX_TRAIN_ROWS, len(X_train)), replace=False)
        predicted = normalise(fit_model(model_name, X_train[chosen], guard_shares[chosen])(X_sample), guard_rolling)
        model_df = guard_sample_df.drop(ROLLING).hstack(pl.DataFrame(predicted, schema=ROLLING))
        imputed_df = impute_hidden(model_df)
        name = f"{model_name}_{set_name}"
        model_scores.append(score_variant(name, GUARD, imputed_df.filter(pl.col(HIDDEN)), IMPUTED, TRUTH, imputed_df))
        print("scored", name, "on hidden months")

## Save the scores

Combines the baselines from notebook 1 with every model variant, in long format, and writes them.

In [ ]:
model_scores_df = pl.concat(model_scores)
scores_df = pl.concat([read_result("baseline_scores.parquet"), model_scores_df])
write_result(model_scores_df, "model_scores.parquet")
write_result(scores_df, "all_scores.parquet")
write_result(pl.concat(trend_series), "trend_series.parquet")
print(scores_df.group_by(VARIANT, TEST).agg(pl.len().alias("scores")).sort(VARIANT, TEST))

## Summary and win rule

One row per variant. `gain_pp` is as-is held-out headline error minus the variant's. `folds_better` counts folds where the variant beats as-is (needs all 5).
`trend_ok` and `guard_ok` mean the variant's trend error and hidden-months headline error are no worse than as-is. `win` needs `gain_pp >= WIN_PP`, all 5 folds, and both guards.
Gradient boosting never wins (ceiling only). Jumpiness is reported, not a rule: `jumpiness_flag` marks a rise in workplace jumpiness of more than
`JUMPINESS_FLAG_FACTOR` times the percentage error gain.

In [ ]:
def lookup(test: str, level: str, metric: str, name: str, group: str = "all") -> pl.DataFrame:
    return scores_df.filter(
        (pl.col(TEST) == test) & (pl.col(LEVEL) == level) & (pl.col(METRIC) == metric) & (pl.col(GROUP) == group)
    ).select(VARIANT, pl.col(VALUE).alias(name))


summary_df = scores_df.select(VARIANT).unique(maintain_order=True)
for test, level, metric, name in [
    (HELD_OUT, "headline", MAE_PP, "headline_pp"),
    (HELD_OUT, "headline_cssr", MAE_PP, "cssr_pp"),
    (HELD_OUT, "aggregate", MAE_PP, "aggregate_pp"),
    (HELD_OUT, "trend", TREND_MAE_PP, "trend_pp"),
    (HELD_OUT, "jumpiness_workplace", MEAN_CHANGE, "jumpiness_workplace"),
    (HELD_OUT, "jumpiness_aggregate", MEAN_CHANGE, "jumpiness_aggregate"),
    (GUARD, "headline", MAE_PP, "guard_headline_pp"),
]:
    summary_df = summary_df.join(lookup(test, level, metric, name), on=VARIANT, how="left")
fold_columns = [f"fold_{fold}_pp" for fold in range(N_FOLDS)]
for fold, name in enumerate(fold_columns):
    summary_df = summary_df.join(lookup(HELD_OUT, "headline_fold", MAE_PP, name, group=str(fold)), on=VARIANT, how="left")

as_is = summary_df.filter(pl.col(VARIANT) == "as_is").row(0, named=True)
is_candidate = ~pl.col(VARIANT).is_in(BASELINES) & ~pl.col(VARIANT).str.starts_with(CEILING_MODEL)
gain_pp = as_is["headline_pp"] - pl.col("headline_pp")
summary_df = (
    summary_df.with_columns(
        gain_pp.alias("gain_pp"),
        pl.sum_horizontal((pl.col(name) < as_is[name]).cast(pl.Int32) for name in fold_columns).alias("folds_better"),
        (pl.col("trend_pp") <= as_is["trend_pp"]).alias("trend_ok"),
        (pl.col("guard_headline_pp") <= as_is["guard_headline_pp"]).alias("guard_ok"),
        ((pl.col("jumpiness_workplace") / as_is["jumpiness_workplace"] - 1) * 100).alias("jumpiness_change_pct"),
        (gain_pp / as_is["headline_pp"] * 100).alias("gain_pct"),
    )
    .with_columns(
        (is_candidate & (pl.col("gain_pp") >= WIN_PP) & (pl.col("folds_better") == N_FOLDS) & pl.col("trend_ok") & pl.col("guard_ok")).alias("win"),
        (
            is_candidate
            & (pl.col("jumpiness_change_pct") > 0)
            & (pl.col("jumpiness_change_pct") > JUMPINESS_FLAG_FACTOR * pl.col("gain_pct").clip(lower_bound=0))
        ).alias("jumpiness_flag"),
    )
)
print(summary_df.select(VARIANT, "headline_pp", "gain_pp", "folds_better", "trend_ok", "guard_ok", "win", "jumpiness_workplace", "jumpiness_change_pct", "jumpiness_flag").sort("headline_pp"))

## Verdict

States whether any candidate wins, which features helped (each model's single-feature gain over its own no-feature version), the best gradient boosting ceiling,
and the jumpiness flags: notable degradation, and ties (candidates within `TIE_PP` of the best headline error, where the steadier one is preferred).

In [ ]:
winners_df = summary_df.filter(pl.col("win"))
print("WINNERS:", winners_df[VARIANT].to_list() or "none: no pipeline change")

candidates_df = summary_df.filter(is_candidate)
best_pp = candidates_df["headline_pp"].min()
print("\nTies with the best candidate (steadier is preferred):")
print(candidates_df.filter(pl.col("headline_pp") <= best_pp + TIE_PP).select(VARIANT, "headline_pp", "jumpiness_workplace", "jumpiness_change_pct").sort("jumpiness_workplace"))
print("\nJumpiness flags (notable degradation relative to the gain):")
print(summary_df.filter(pl.col("jumpiness_flag")).select(VARIANT, "gain_pct", "jumpiness_change_pct"))

print("\nFeature effect: headline error gain over the same model with no feature (pp, positive = helps)")
for model_name in MODELS_PER_SET:
    base_pp = summary_df.filter(pl.col(VARIANT) == f"{model_name}_base")["headline_pp"].item()
    print(model_name)
    print(
        summary_df.filter(pl.col(VARIANT).str.starts_with(f"{model_name}_") & (pl.col(VARIANT) != f"{model_name}_base"))
        .select(pl.col(VARIANT), (base_pp - pl.col("headline_pp")).alias("gain_over_base_pp"), "folds_better")
    )
print("\nAccuracy ceiling:")
print(summary_df.filter(pl.col(VARIANT).str.starts_with(CEILING_MODEL)).select(VARIANT, "headline_pp", "gain_pp", "trend_pp", "guard_headline_pp"))

## Extra tables

- **Per fold:** `fold_*_pp` above. **Dormancy pre/post:** row-level error where dormancy was not yet recorded versus recorded. **Size band:** row-level error by band.
- **CSSR:** `cssr_pp` in the summary (service x CSSR x role x year, worker-weighted).
- **National trend:** `trend_series.parquet` holds the mean predicted and known share per date and status for every variant, for plotting.

In [ ]:
def split_table(level: str) -> pl.DataFrame:
    return (
        scores_df.filter((pl.col(TEST) == HELD_OUT) & (pl.col(LEVEL) == level) & (pl.col(METRIC) == MAE_PP))
        .pivot(on=GROUP, index=VARIANT, values=VALUE)
        .sort(VARIANT)
    )


print("Row-level error (pp) by size band")
print(split_table("share_size_band"))
print("Row-level error (pp) by dormancy recorded or not")
print(split_table("share_dormancy"))
print("Headline error (pp) by CSSR and region")
print(summary_df.select(VARIANT, "headline_pp", "cssr_pp", "aggregate_pp").sort("headline_pp"))